In [3]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

# ---- load everything ----
CAT_PATH = '/kaggle/input/datasets/saiymasarmin/catboost-all-10/'
XG_PATH = '/kaggle/input/datasets/saiymasarmin/xgboost-all-10/'
xgb_oof = pd.read_csv(XG_PATH + 'xgboost_oof.csv')
cat_oof = pd.read_csv(CAT_PATH + 'catboost_oof.csv')
xgb_test = pd.read_csv(XG_PATH + 'xgboost_test_probs.csv')
cat_test = pd.read_csv(CAT_PATH + 'catboost_test_probs.csv')

# Merge on id rather than assuming row order matches between the two files
oof = xgb_oof.merge(cat_oof, on='id', suffixes=('_xgb', '_cat'))
assert (oof['true_label_xgb'] == oof['true_label_cat']).all(), "true labels don't match between the two OOF files!"
oof['true_label'] = oof['true_label_xgb']

test = xgb_test.merge(cat_test, on='id', suffixes=('_xgb', '_cat'))

y_true = oof['true_label'].values
xgb_oof_p = oof['oof_proba_xgb'].values
cat_oof_p = oof['oof_proba_cat'].values


# ---- combination methods ----
def weighted_average(p1, p2, w):
    """w = weight on p2 (CatBoost); (1-w) on p1 (XGBoost)."""
    return (1 - w) * p1 + w * p2

def rank_average(p1, p2):
    """Convert each to ranks, then average the ranks. Sidesteps any
    scale/calibration mismatch between the two models."""
    r1 = rankdata(p1)
    r2 = rankdata(p2)
    return (r1 + r2) / 2

def logit_average(p1, p2, eps=1e-6):
    """Average in log-odds space, then convert back to a probability."""
    p1c = np.clip(p1, eps, 1 - eps)
    p2c = np.clip(p2, eps, 1 - eps)
    logit1 = np.log(p1c / (1 - p1c))
    logit2 = np.log(p2c / (1 - p2c))
    avg_logit = (logit1 + logit2) / 2
    return 1 / (1 + np.exp(-avg_logit))


# ---- experiment: try everything, score each on OOF ----
results = []

results.append(("xgboost_alone", roc_auc_score(y_true, xgb_oof_p)))
results.append(("catboost_alone", roc_auc_score(y_true, cat_oof_p)))
results.append(("simple_average_50_50", roc_auc_score(y_true, weighted_average(xgb_oof_p, cat_oof_p, 0.5))))
results.append(("rank_average", roc_auc_score(y_true, rank_average(xgb_oof_p, cat_oof_p))))
results.append(("logit_average", roc_auc_score(y_true, logit_average(xgb_oof_p, cat_oof_p))))

best_weight, best_weight_auc = None, -1
# for w in np.arange(0.0, 1.01, 0.01):
for w in np.arange(.74, .80, 0.001):
    auc = roc_auc_score(y_true, weighted_average(xgb_oof_p, cat_oof_p, w))
    results.append((f"weighted_avg_w={w:.6f}", auc))
    if auc > best_weight_auc:
        best_weight, best_weight_auc = w, auc

results_df = pd.DataFrame(results, columns=["method", "oof_roc_auc"]).sort_values("oof_roc_auc", ascending=False)
print(results_df.to_string(index=False))
print(f"\nBest single weighted-average weight on CatBoost: w={best_weight:.6f}  (OOF ROC-AUC = {best_weight_auc:.6f})")


# ---- build final submission using whichever method won ----
# Change this line to match whichever method actually won above
final_test_probs = weighted_average(test['test_proba_xgb'].values, test['test_proba_cat'].values, best_weight)
# Alternatives if rank_average or logit_average wins instead:
# final_test_probs = rank_average(test['test_proba_xgb'].values, test['test_proba_cat'].values)
# final_test_probs = logit_average(test['test_proba_xgb'].values, test['test_proba_cat'].values)

submission = pd.DataFrame({
    'id': test['id'],
    'addicted_label': final_test_probs
})
submission.to_csv('submission.csv', index=False)
print(submission.head())

                 method  oof_roc_auc
weighted_avg_w=0.765000     0.967617
weighted_avg_w=0.762000     0.967617
weighted_avg_w=0.766000     0.967617
weighted_avg_w=0.764000     0.967617
weighted_avg_w=0.763000     0.967617
weighted_avg_w=0.759000     0.967617
weighted_avg_w=0.761000     0.967617
weighted_avg_w=0.760000     0.967617
weighted_avg_w=0.767000     0.967617
weighted_avg_w=0.768000     0.967617
weighted_avg_w=0.758000     0.967617
weighted_avg_w=0.769000     0.967617
weighted_avg_w=0.757000     0.967617
weighted_avg_w=0.770000     0.967617
weighted_avg_w=0.756000     0.967617
weighted_avg_w=0.771000     0.967617
weighted_avg_w=0.755000     0.967617
weighted_avg_w=0.772000     0.967617
weighted_avg_w=0.754000     0.967617
weighted_avg_w=0.773000     0.967617
weighted_avg_w=0.774000     0.967617
weighted_avg_w=0.753000     0.967617
weighted_avg_w=0.752000     0.967617
weighted_avg_w=0.775000     0.967617
weighted_avg_w=0.776000     0.967617
weighted_avg_w=0.751000     0.967617
w